# PyExplain — Live demo (7B) on Colab

This loads the published **PyExplain 7B** model and launches a small web app
with a **public link** anyone can use — running on Colab's free GPU.

## Steps
1. **Set the runtime to GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run **Cell 1** (installs libraries, ~1 min).
3. Run **Cell 2** (loads the model + starts the demo). The first load takes a
   few minutes while the 7B downloads.
4. When it finishes, it prints a public link like `https://xxxxx.gradio.live`
   — open it or share it. It stays live **while this Colab is running**.

> Keep this tab open. When you close Colab (or it times out), the link goes dead — just re-run to get a new one.

### Cell 1 — Install libraries

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes gradio

### Cell 2 — Load the 7B and launch the demo

In [ ]:
import torch, gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER = "AyushPatel28/PyExplain-qwen-coder-7b"

SYSTEM_PROMPT = (
    "You are an expert Python engineer explaining code to someone who has ZERO "
    "knowledge of Python or programming. Explain everything in the simplest, "
    "plainest everyday language so a complete beginner can fully understand. "
    "Use the proper programming terms, but the instant you use a term, explain "
    "in plain words what it means. Start with one plain sentence on what the "
    "code does overall, then explain the actual code part by part. IMPORTANT: "
    "do NOT include any worked example, do NOT trace the code with numbers, and "
    "do NOT calculate any result by hand — explain only in words."
)

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER).eval()
for a in ("temperature", "top_p", "top_k"):
    if hasattr(model.generation_config, a):
        setattr(model.generation_config, a, None)

def explain(code):
    if not (code or "").strip():
        return "Paste some Python code first."
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Explain the following Python code:\n```python\n{code}\n```"}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(p, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inp, max_new_tokens=350, do_sample=False, repetition_penalty=1.15,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

gr.Interface(
    fn=explain,
    inputs=gr.Code(language="python", label="Your Python code"),
    outputs=gr.Textbox(label="Explanation", lines=12),
    title="\ud83d\udc0d PyExplain \u2014 Python code, explained",
    description="Fine-tuned Qwen2.5-Coder-7B (QLoRA). Paste Python code and get a plain-English explanation.",
    examples=[["def reverse(s):\n    return s[::-1]"],
              ["squares = [x * x for x in range(5)]"]],
    flagging_mode="never",
).launch(share=True)